<!-- # The Bayesian Finite Element Method in Inverse Problems: Pullout Test

This notebook is associated with section 3.1 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)). -->
# Convergence

In [ ]:
# general imports
import os
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from scipy.stats.qmc import LatinHypercube
from sksparse.cholmod import cholesky
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon
import seaborn as sns

from myjivex.util import QuickViewer, ElemViewer

from fem.jive import CJiveRunner
from fem.meshing import (
    mesh_interval_with_line2,
    mesh_rectangle_with_tri3,
    create_unit_mass_matrix,
    create_bboxes,
)

from experiments.reproduction.theory.props import get_fem_props
from experiments.reproduction.theory import convergence as util

In [ ]:
dimensionality = 2  # 1D or 2D problem
d = 0.1  # interval size of adjoint source term
norm = "energy"  # options: 'energy' or 'l2'
n_lhc = 9  # number of latin hypercube samples (must be a square of a prime)
n_dense = 256  # number of dense sampling points (for plotting)
save_plots = False  # output figures

In [ ]:
# matplotlib settings
plt.rc("text", usetex=True)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]
plt.rcParams["font.size"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["text.latex.preamble"] = r"\usepackage{xfrac}"

## Convergence in 1D

In [ ]:
if dimensionality == 1:

    def true_f_solution(coord):
        return util.true_f_solution_1d(coord)

    def true_f_strain(coord):
        return util.true_f_strain_1d(coord)

    def true_f_source(coord):
        return util.true_f_source_1d(coord)

elif dimensionality == 2:

    def true_f_solution(coord):
        return util.true_f_solution_2d(coord)

    def true_f_strain(coord):
        return util.true_f_strain_2d(coord)

    def true_f_source(coord):
        return util.true_f_source_2d(coord)

else:
    assert False

In [ ]:
# true solution
x = np.linspace(0, 1, n_dense)
y = np.linspace(0, 1, n_dense)

if dimensionality == 1:
    domain_coords = x.reshape(-1, 1)
elif dimensionality == 2:
    X, Y = np.meshgrid(x, y)
    domain_coords = np.column_stack([X.ravel(), Y.ravel()])

ndom = len(domain_coords)
u = np.zeros(ndom)
eps = np.zeros((ndom, dimensionality))
f = np.zeros(ndom)

for i, coord in enumerate(domain_coords):
    u[i] = true_f_solution(coord)
    eps[i] = true_f_strain(coord)
    f[i] = true_f_source(coord)

if dimensionality == 1:
    fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
    axs[0].plot(x, u)
    axs[1].plot(x, eps)
    axs[2].plot(x, f)
    plt.show()

elif dimensionality == 2:
    cmap = LinearSegmentedColormap.from_list(
        "rocket_truncated", sns.cm.rocket(np.linspace(0.3, 1.0, 256))
    )

    contourf_kwargs = {
        "levels": 1000,
        "cmap": cmap,
    }

    contour_kwargs = {
        "colors": "0.3",
        "linewidths": 0.5,
        "linestyles": "solid",
        "alpha": 0.3,
        "levels": 20,
    }

    true_fields = [u, eps[..., 0], eps[..., 1], f]
    true_names = ["u", "eps-x", "eps-y", "f"]

    if not save_plots:
        fig, axs = plt.subplots(ncols=4, figsize=(12, 3))

    for i, field in enumerate(true_fields):
        if save_plots:
            fig, ax = plt.subplots(figsize=(3, 3))
        else:
            ax = axs[i]

        ax.contourf(X, Y, field.reshape(n_dense, n_dense), **contourf_kwargs)
        ax.contour(X, Y, field.reshape(n_dense, n_dense), **contour_kwargs)
        ax.set_xticks([0.0, 1.0])
        ax.set_yticks([0.0, 1.0])
        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$y$")

        if save_plots:
            fname = "true_{}_2d.png".format(true_names[i])
            fname = os.path.join("plots", fname)
            os.makedirs(os.path.dirname(fname), exist_ok=True)
            plt.savefig(fname, dpi=600, bbox_inches="tight")
            plt.show()

    if not save_plots:
        plt.show()

In [ ]:
if dimensionality == 1:
    nodes, elems = mesh_interval_with_line2(n=8)
elif dimensionality == 2:
    nodes, elems = mesh_rectangle_with_tri3(n=8)

props = get_fem_props(dimensionality)
jive = CJiveRunner(props, elems=elems)
globdat = jive()

In [ ]:
if dimensionality == 1:
    x_h = nodes.get_coords().flatten()
    u_h = globdat["state0"]

    fig, ax = plt.subplots()
    ax.plot(x_h, u_h)
    plt.show()

elif dimensionality == 2:
    quickviewer_kwargs = {
        # "linewidth": 0.2,
        "comp": 0,
        "colormap": cmap,
        "colorbar": False,
    }
    elemviewer_kwargs = {
        # "linewidth": 0.2,
        "comp": 0,
        "colormap": cmap,
        "colorbar": False,
    }
    meshviewer_kwargs = {
        "meshonly": True,
        "linewidth": 0.2,
        "comp": 0,
        "colormap": cmap,
        "colorbar": False,
    }

    strains = util.calc_strains_per_element(globdat)

    fem_fields = [
        globdat["state0"],
        strains[:, 0],
        strains[:, 1],
        globdat["extForce"],
        "mesh",
    ]
    fem_fnames = [
        "fem_u_2d.png",
        "fem_eps-x_2d.png",
        "fem_eps-y_2d.png",
        "fem_f_2d.png",
        "fem_mesh-2d.pdf",
    ]

    if not save_plots:
        fig, axs = plt.subplots(ncols=4, figsize=(12, 3))

    for i, field in enumerate(fem_fields):
        if save_plots:
            fig, ax = plt.subplots(figsize=(3, 3))
        else:
            ax = axs[i]

        if isinstance(field, str):
            if field == "mesh":
                dummy_field = np.zeros(len(globdat["nodeSet"]))
                QuickViewer(dummy_field, globdat, ax=ax, **meshviewer_kwargs)
            else:
                assert False

        else:
            if len(field) == len(globdat["nodeSet"]):
                QuickViewer(field, globdat, ax=ax, **quickviewer_kwargs)
            elif len(field) == len(globdat["elemSet"]):
                ElemViewer(field, globdat, ax=ax, **elemviewer_kwargs)
            else:
                assert False

        ax.set_aspect("equal", adjustable="box")
        ax.set_axis_on()
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.0])
        ax.set_xticks([0.0, 1.0])
        ax.set_yticks([0.0, 1.0])
        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$y$")

        if save_plots:
            fname = os.path.join("plots", fem_fnames[i])
            os.makedirs(os.path.dirname(fname), exist_ok=True)
            plt.savefig(fname, dpi=600, bbox_inches="tight")
            plt.show()

    if not save_plots:
        plt.show()

### 1D prior convergence with reference mesh refinement

In [ ]:
if dimensionality == 1:

    def true_g_solution(coord, a, b):
        return util.true_g_solution_1d(coord, a, b)

    def true_g_strain(coord, a, b):
        return util.true_g_strain_1d(coord, a, b)

    def true_g_source(coord, a, b):
        return util.true_g_source_1d(coord, a, b)

elif dimensionality == 2:

    def true_g_solution(coord, a, b):
        return util.true_g_solution_2d(coord, a, b)

    def true_g_strain(coord, a, b):
        return util.true_g_strain_2d(coord, a, b)

    def true_g_source(coord, a, b):
        return util.true_g_source_2d(coord, a, b)

else:
    assert False

In [ ]:
lhc = LatinHypercube(d=dimensionality, seed=0, strength=2)
ms = lhc.random(n=n_lhc) * (1.0 - d) + 0.5 * d

if dimensionality == 1:
    fig, ax = plt.subplots()

    for i, (j, m) in enumerate(sorted(enumerate(ms), key=lambda im: im[1])):
        a = m[0] - 0.5 * d
        b = m[0] + 0.5 * d
        dy = 0.02

        lb = (a, 0.4 * dy * (-1) ** i)
        lc = (a, 0.7 * dy * (-1) ** i)
        lt = (a, 1.0 * dy * (-1) ** i)
        rb = (b, 0.4 * dy * (-1) ** i)
        rc = (b, 0.7 * dy * (-1) ** i)
        rt = (b, 1.0 * dy * (-1) ** i)

        ax.plot([lb[0], lt[0]], [lb[1], lt[1]], color="0.5", linewidth=1.0)
        ax.plot([lc[0], rc[0]], [lc[1], rc[1]], color="0.5", linewidth=1.0)
        ax.plot([rb[0], rt[0]], [rb[1], rt[1]], color="0.5", linewidth=1.0)

        if j == 0:
            ax.text(m[0], 2.0 * dy * (-1) ** i, "A", ha="center", va="center")

    ax.set_aspect("equal")
    ax.spines["bottom"].set_position("center")
    ax.spines["left"].set_color("none")
    ax.spines["right"].set_color("none")
    ax.spines["top"].set_color("none")
    ax.set_yticks([])
    ax.set_xticks([0.0, 1.0])
    plt.show()

elif dimensionality == 2:
    fig, ax = plt.subplots()

    for j, m in enumerate(ms):
        iso_coords = np.array([[-1.0, -1.0], [1.0, -1.0], [1.0, 1.0], [-1.0, 1.0]])
        coords = m + iso_coords * d / 2
        patch = Polygon(coords, fill=False, color="0.5", linewidth=1.0)
        ax.add_patch(patch)

        if j == 0:
            ax.text(m[0], m[1], "A", ha="center", va="center")

    ax.set_yticks([0.0, 1.0])
    ax.set_xticks([0.0, 1.0])
    ax.set_aspect("equal")
    plt.show()

In [ ]:
ug = np.zeros(ndom)
epsg = np.zeros((ndom, dimensionality))
g = np.zeros(ndom)

a = ms[0] - 0.5 * d
b = ms[0] + 0.5 * d

for i, coord in enumerate(domain_coords):
    ug[i] = true_g_solution(coord, a, b)
    epsg[i] = true_g_strain(coord, a, b)
    g[i] = true_g_source(coord, a, b)

if dimensionality == 1:
    fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
    axs[0].plot(x, ug)
    axs[1].plot(x, epsg)
    axs[2].plot(x, g)
    plt.show()

elif dimensionality == 2:
    true_fields = [ug, epsg[..., 0], epsg[..., 1], g]
    true_names = ["ug", "epsg-x", "epsg-y", "g"]

    if not save_plots:
        fig, axs = plt.subplots(ncols=4, figsize=(12, 3))

    for i, field in enumerate(true_fields):
        if save_plots:
            fig, ax = plt.subplots(figsize=(3, 3))
        else:
            ax = axs[i]

        ax.contourf(X, Y, field.reshape(n_dense, n_dense), **contourf_kwargs)
        ax.contour(X, Y, field.reshape(n_dense, n_dense), **contour_kwargs)
        ax.set_xticks([0.0, 1.0])
        ax.set_yticks([0.0, 1.0])
        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$y$")

        if save_plots:
            fname = "true_{}_2d.png".format(true_names[i])
            fname = os.path.join("plots", fname)
            os.makedirs(os.path.dirname(fname), exist_ok=True)
            plt.savefig(fname, dpi=600, bbox_inches="tight")
            plt.show()

    if not save_plots:
        plt.show()

In [ ]:
def true_gg_inner_product(a, b, *, norm):

    if dimensionality == 1:
        ax, bx = a[0], b[0]

        if norm == "energy":
            dux_dx_norm = util.true_g_ux_H1_seminorm(ax, bx)
            return dux_dx_norm**2
        elif norm == "l2":
            ux_norm = util.true_g_ux_H0_norm(ax, bx)
            return ux_norm**2
        else:
            assert False

    elif dimensionality == 2:
        ax, ay = a
        bx, by = b

        if norm == "energy":
            ux_norm = util.true_g_ux_H0_norm(ax, bx)
            uy_norm = util.true_g_uy_H0_norm(ay, by)
            dux_dx_norm = util.true_g_ux_H1_seminorm(ax, bx)
            duy_dy_norm = util.true_g_uy_H1_seminorm(ay, by)
            return dux_dx_norm**2 * uy_norm**2 + ux_norm**2 * duy_dy_norm**2

        elif norm == "l2":
            ux_norm = util.true_g_ux_H0_norm(ax, bx)
            uy_norm = util.true_g_uy_H0_norm(ay, by)
            return ux_norm**2 * uy_norm**2

        else:
            assert False

    else:
        assert False

In [ ]:
def fem_gg_inner_product(a, b, *, norm, globdat):
    g = util.fem_g_source(a, b, globdat=globdat)
    ug = util.fem_g_solution(a, b, globdat=globdat, g=g)

    if norm == "energy":
        return ug @ g
    elif norm == "l2":
        M = globdat["matrix2"]
        return ug @ M @ ug
    else:
        assert False

In [ ]:
def true_prior_std(a, b, *, norm):
    return np.sqrt(true_gg_inner_product(a, b, norm=norm))


def fem_prior_std(a, b, *, norm, globdat_ref):
    return np.sqrt(fem_gg_inner_product(a, b, norm=norm, globdat=globdat_ref))


def true_posterior_std(a, b, *, norm, globdat_obs):
    std_prior = true_prior_std(a, b, norm=norm)
    var_downdate = fem_gg_inner_product(a, b, norm=norm, globdat=globdat_obs)
    return np.sqrt(std_prior**2 - var_downdate)


def fem_posterior_std(a, b, *, norm, globdat_obs, globdat_ref):
    std_prior_h = fem_prior_std(a, b, norm=norm, globdat_ref=globdat_ref)
    var_downdate = fem_gg_inner_product(a, b, norm=norm, globdat=globdat_obs)
    return np.sqrt(std_prior_h**2 - var_downdate)


def posterior_mean(a, b, *, norm, globdat_obs):
    obs_nodes = globdat_obs["nodeSet"]
    obs_elems = globdat_obs["elemSet"]

    g = util.fem_g_source(a, b, globdat=globdat_obs)

    if norm == "energy":
        Kc = globdat_obs["Kc"]
        fc = globdat_obs["extForce"].copy()
        cdofs = globdat_obs["constraints"].get_constraints()[0]

        fc[cdofs] = 0.0
        uf = Kc.solve_A(fc)
        return g @ uf

    elif norm == "l2":
        Mc = globdat_obs["Mc"]
        fc = globdat_obs["extForce"].copy()
        cdofs = globdat_obs["constraints"].get_constraints()[0]
        fc[cdofs] = 0.0
        fhat = Mc.solve_A(fc)

        dofs = globdat_obs["dofSpace"]
        shape = globdat_obs["shape"]

        ug = util.fem_quadrature(
            true_g_solution,
            args={"a": a, "b": b},
            mesh=obs_mesh,
            dofs=dofs,
            shape=shape,
        )

        return ug @ fhat
    else:
        assert False

In [ ]:
if dimensionality == 1:
    ns = 2 ** np.arange(2, 16)
elif dimensionality == 2:
    ns = 2 ** np.arange(1, 9)

meshes = []
globdats = []

for n in ns:
    if dimensionality == 1:
        mesh = mesh_interval_with_line2(n=n)
    elif dimensionality == 2:
        mesh = mesh_rectangle_with_tri3(n=n)

    nodes, elems = mesh
    props = get_fem_props(dimensionality)
    jive = CJiveRunner(props, elems=elems)
    globdat = jive()

    dofs = globdat["dofSpace"]
    shape = globdat["shape"]
    cdofs = globdat["constraints"].get_constraints()[0]

    Kc = globdat["matrix0"].copy()
    Kc[:, cdofs] *= 0.0
    Kc[cdofs, :] *= 0.0
    Kc[cdofs, cdofs] = 1.0
    globdat["Kc"] = cholesky(csc_matrix(Kc))

    M = create_unit_mass_matrix(elems, dofs, shape, sparse=True, lumped=False)
    Mc = M.copy()
    Mc[:, cdofs] *= 0.0
    Mc[cdofs, :] *= 0.0
    Mc[cdofs, cdofs] = 1.0
    globdat["matrix2"] = M
    globdat["Mc"] = cholesky(csc_matrix(Mc))

    globdat["bboxes"] = create_bboxes(elems)

    meshes.append(mesh)
    globdats.append(globdat)

In [ ]:
prior_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = meshes[i]
    globdat_ref = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_prior = true_prior_std(a, b, norm=norm)
        sigma_prior_h = fem_prior_std(a, b, norm=norm, globdat_ref=globdat_ref)
        prior_ref_W2_distances[i, j] = abs(sigma_prior - sigma_prior_h)

In [ ]:
def lim_to_ticks(lim):
    ltick = np.ceil(np.log10(lim[0]))
    utick = np.floor(np.log10(lim[1]))
    return 10 ** np.arange(ltick, utick + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))

for j, m in enumerate(ms):
    W2_dist = prior_ref_W2_distances[:, j]
    ax.loglog(ns, W2_dist, color="0.5", alpha=0.5)

ax.set_aspect("equal")

xlim = ax.get_xlim()
ylim = ax.get_ylim()

if dimensionality == 1:
    xlim = (2e0, 5e4)
elif dimensionality == 2:
    xlim = (1e0, 5e2)

    if norm == "energy":
        ylim = (1e-6, 1e-1)
    elif norm == "l2":
        ylim = (1e-7, 1e-2)

ax.set_xticks(lim_to_ticks(xlim))
ax.set_yticks(lim_to_ticks(ylim))
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r"$\tilde{h}^{-1}$")
ax.set_ylabel(r"$W_2(\nu,\tilde{\nu})$")

ax.grid()

if save_plots:
    fname = "convergence_prior_reference_{}d_{}.pdf"
    fname = fname.format(dimensionality, norm)
    fname = os.path.join("plots", fname)
    os.makedirs(os.path.dirname(fname), exist_ok=True)
    plt.savefig(fname, bbox_inches="tight")

plt.show()

### 1D posterior convergence with reference mesh refinement

In [ ]:
if dimensionality == 1:
    obs_mesh = mesh_interval_with_line2(n=4)
elif dimensionality == 2:
    obs_mesh = mesh_rectangle_with_tri3(n=2)

obs_nodes, obs_elems = obs_mesh
props = get_fem_props(dimensionality)
jive = CJiveRunner(props, elems=obs_elems)
globdat_obs = jive()

dofs = globdat_obs["dofSpace"]
shape = globdat_obs["shape"]
cdofs = globdat_obs["constraints"].get_constraints()[0]

Kc = globdat_obs["matrix0"].copy()
Kc[:, cdofs] *= 0.0
Kc[cdofs, :] *= 0.0
Kc[cdofs, cdofs] = 1.0
globdat_obs["Kc"] = cholesky(csc_matrix(Kc))

M_obs = create_unit_mass_matrix(obs_elems, dofs, shape, sparse=True, lumped=False)
globdat_obs["matrix2"] = M_obs
Mc = M_obs.copy()
Mc[:, cdofs] *= 0.0
Mc[cdofs, :] *= 0.0
Mc[cdofs, cdofs] = 1.0
globdat_obs["Mc"] = cholesky(csc_matrix(Mc))

globdat_obs["bboxes"] = create_bboxes(obs_elems)

post_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = meshes[i]
    globdat_ref = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_post = true_posterior_std(a, b, norm=norm, globdat_obs=globdat_obs)
        sigma_post_h = fem_posterior_std(
            a, b, norm=norm, globdat_obs=globdat_obs, globdat_ref=globdat_ref
        )
        post_ref_W2_distances[i, j] = abs(sigma_post - sigma_post_h)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))

for j, m in enumerate(ms):
    W2_dist = post_ref_W2_distances[:, j]
    ax.loglog(ns, W2_dist, color="0.5", alpha=0.5)

ax.set_aspect("equal")

xlim = ax.get_xlim()
ylim = ax.get_ylim()

if dimensionality == 1:
    xlim = (2e0, 5e4)
elif dimensionality == 2:
    xlim = (1e0, 5e2)

    if norm == "energy":
        ylim = (1e-6, 1e-1)
    elif norm == "l2":
        ylim = (1e-7, 1e-2)

ax.set_xticks(lim_to_ticks(xlim))
ax.set_yticks(lim_to_ticks(ylim))
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r"$\tilde{h}^{-1}$")
ax.set_ylabel(r"$W_2(\nu_*,\tilde{\nu}_*)$")

ax.grid()

if save_plots:
    fname = "convergence_posterior_reference_{}d_{}.pdf"
    fname = fname.format(dimensionality, norm)
    fname = os.path.join("plots", fname)
    os.makedirs(os.path.dirname(fname), exist_ok=True)
    plt.savefig(fname, bbox_inches="tight")

plt.show()

### 1D posterior convergence with observation mesh refinement

In [ ]:
def true_qoi(a, b):

    if dimensionality == 1:
        # <u, g> = int_0^1 int_0^1 u_x * -d^2ug_x/dx^2 dx
        #        = int_0^1 du_x/dx dug_x/dx dx
        ax, bx = a[0], b[0]
        I = util.true_integral_dux_dx_dugx_dx(ax, bx)
        return I

    elif dimensionality == 2:
        # <u, g> = int_0^1 int_0^1 u_x * u_y * -(d^2ug_x/dx^2 * ug_y + ug_x * d^2ug_y/dy^2) dx dy
        #        = int_0^1 u_x ug_x dx * int_0^1 du_y/dy dug_y/dy dy + int_0^1 du_x/dx dug_x/dx dx * int_0^1 u_y ug_y dy
        #        = I1                  * I2                          + I3                          * I4
        ax, ay = a
        bx, by = b

        I1 = util.true_integral_ux_ugx(ax, bx)
        I2 = util.true_integral_duy_dy_dugy_dy(ay, by)
        I3 = util.true_integral_dux_dx_dugx_dx(ax, bx)
        I4 = util.true_integral_uy_ugy(ay, by)

        I = I1 * I2 + I3 * I4
        return I

In [ ]:
post_obs_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    obs_mesh = meshes[i]
    globdat_obs = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        mu_post = posterior_mean(a, b, norm=norm, globdat_obs=globdat_obs)
        mu_true = true_qoi(a, b)
        sigma_post = true_posterior_std(a, b, norm=norm, globdat_obs=globdat_obs)
        post_obs_W2_distances[i, j] = np.sqrt((mu_true - mu_post) ** 2 + sigma_post**2)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))
ax.set_aspect("equal")

for j, m in enumerate(ms):
    W2_dist = post_obs_W2_distances[:, j]
    ax.loglog(ns, W2_dist, color="0.5", alpha=0.5)

xlim = ax.get_xlim()
ylim = ax.get_ylim()

if dimensionality == 1:
    xlim = (2e0, 5e4)
elif dimensionality == 2:
    xlim = (1e0, 5e2)

    if norm == "energy":
        ylim = (1e-4, 1e-1)
    elif norm == "l2":
        ylim = (1e-5, 1e-1)

ax.set_xticks(lim_to_ticks(xlim))
ax.set_yticks(lim_to_ticks(ylim))
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r"$h^{-1}$")
ax.set_ylabel(r"$W_2(\delta, \nu_*)$")

ax.grid()

if save_plots:
    fname = "convergence_posterior_observation_{}d_{}.pdf"
    fname = fname.format(dimensionality, norm)
    fname = os.path.join("plots", fname)
    os.makedirs(os.path.dirname(fname), exist_ok=True)
    plt.savefig(fname, bbox_inches="tight")

plt.show()